In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

print('Libraries loaded successfully.')

In [ ]:
# Load dataset
df = pd.read_csv('marketing_data.csv')   

print('Shape (rows x cols):', df.shape)
df.head()

In [ ]:
# Basic summary statistics
df.describe().round(2)

In [ ]:
# Missing-value audit
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
print(pd.concat([missing.rename('Missing Count'), pct.rename('Missing %')], axis=1))

In [ ]:
# Drop rows with any NaN - missingness is < 0.3 % of rows for each column,
# so listwise deletion is appropriate and introduces no meaningful bias.
df_clean = df.dropna().reset_index(drop=True)

print(f'Rows before cleaning : {len(df):,}')
print(f'Rows after cleaning  : {len(df_clean):,}')
print(f'Rows removed         : {len(df) - len(df_clean):,}')

In [ ]:
# Distribution of each variable
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cols = ['TV', 'Radio', 'Social_Media', 'Sales']
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

for ax, col, color in zip(axes, cols, colors):
    ax.hist(df_clean[col], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel('Budget ($000s)' if col != 'Sales' else 'Sales ($000s)')
    ax.set_ylabel('Frequency')

plt.suptitle('Distribution of Marketing Spend & Sales', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots: each channel vs Sales
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
channels = ['TV', 'Radio', 'Social_Media']
colors   = ['#4C72B0', '#55A868', '#C44E52']

for ax, ch, color in zip(axes, channels, colors):
    ax.scatter(df_clean[ch], df_clean['Sales'], alpha=0.25, s=10, color=color)
    m, b = np.polyfit(df_clean[ch], df_clean['Sales'], 1)
    x_line = np.linspace(df_clean[ch].min(), df_clean[ch].max(), 200)
    ax.plot(x_line, m * x_line + b, color='black', linewidth=2, label='OLS fit')
    r = df_clean[[ch, 'Sales']].corr().iloc[0, 1]
    ax.set_title(f'{ch} vs Sales\n(r = {r:.3f})', fontsize=12, fontweight='bold')
    ax.set_xlabel(f'{ch} Budget ($000s)')
    ax.set_ylabel('Sales ($000s)')
    ax.legend(fontsize=9)

plt.suptitle('Marketing Channel vs Sales - Scatter Plots', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heat-map
corr_matrix = df_clean.corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)   # upper triangle only
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
    vmin=-1, vmax=1, ax=ax, linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation with Sales:')
print(corr_matrix['Sales'].sort_values(ascending=False))

In [ ]:
# Prepare X and y
X = sm.add_constant(df_clean['TV'])   # adds the intercept column
y = df_clean['Sales']

# Fit OLS
model = sm.OLS(y, X).fit()

# Full summary
print(model.summary())

In [ ]:
# Regression line plot
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(df_clean['TV'], df_clean['Sales'], alpha=0.2, s=8, color='#4C72B0', label='Observed')

x_range = np.linspace(df_clean['TV'].min(), df_clean['TV'].max(), 300)
X_pred  = sm.add_constant(x_range)
y_pred  = model.predict(X_pred)

ax.plot(x_range, y_pred, color='#C44E52', linewidth=2.5, label=f'OLS fit  (R^2={model.rsquared:.4f})')

ax.set_xlabel('TV Budget ($000s)', fontsize=12)
ax.set_ylabel('Sales ($000s)', fontsize=12)
ax.set_title('TV Spend vs Sales - OLS Regression', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('regression_line.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fitted    = model.fittedvalues
residuals = model.resid
std_resid = (residuals - residuals.mean()) / residuals.std()

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# --- Plot 1: Residuals vs Fitted (Linearity + Homoscedasticity) ---
ax = axes[0, 0]
ax.scatter(fitted, residuals, alpha=0.2, s=8, color='#4C72B0')
ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
lowess_fit = sm.nonparametric.lowess(residuals, fitted, frac=0.3)
ax.plot(lowess_fit[:, 0], lowess_fit[:, 1], color='orange', linewidth=2, label='LOWESS')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted\n(tests Linearity & Homoscedasticity)', fontweight='bold')
ax.legend()

# --- Plot 2: Q-Q Plot (Normality) ---
ax = axes[0, 1]
sm.qqplot(residuals, line='s', ax=ax, alpha=0.3, markersize=3)
ax.set_title('Normal Q-Q Plot\n(tests Normality of Residuals)', fontweight='bold')

# --- Plot 3: Scale-Location (Homoscedasticity) ---
ax = axes[1, 0]
ax.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.2, s=8, color='#55A868')
lowess_sl = sm.nonparametric.lowess(np.sqrt(np.abs(std_resid)), fitted, frac=0.3)
ax.plot(lowess_sl[:, 0], lowess_sl[:, 1], color='orange', linewidth=2, label='LOWESS')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('sqrt|Standardised Residuals|')
ax.set_title('Scale-Location\n(tests Homoscedasticity)', fontweight='bold')
ax.legend()

# --- Plot 4: Residual Histogram (Normality) ---
ax = axes[1, 1]
ax.hist(residuals, bins=50, color='#8172B2', edgecolor='white', alpha=0.85, density=True)
x_norm = np.linspace(residuals.min(), residuals.max(), 300)
ax.plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
        color='red', linewidth=2, label='Normal curve')
ax.set_xlabel('Residual Value')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution\n(tests Normality)', fontweight='bold')
ax.legend()

plt.suptitle('Regression Diagnostic Plots', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('diagnostic_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical tests for assumptions
from statsmodels.stats.diagnostic import het_breuschpagan, linear_harvey_collier
from statsmodels.stats.stattools import jarque_bera

# Normality: Jarque-Bera
jb_stat, jb_p, jb_skew, jb_kurt = jarque_bera(residuals)
print(f'=== Jarque-Bera (Normality) ===')
print(f'  Statistic : {jb_stat:.4f}')
print(f'  p-value   : {jb_p:.4f}')
print(f'  Skewness  : {jb_skew:.4f}  (ideal = 0)')
print(f'  Kurtosis  : {jb_kurt:.4f}  (excess; ideal = 0)')

# Homoscedasticity: Breusch-Pagan
bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(residuals, model.model.exog)
print(f'\n=== Breusch-Pagan (Homoscedasticity) ===')
print(f'  LM Statistic: {bp_lm:.4f}')
print(f'  p-value     : {bp_p:.4f}')

print('\nSee notes in the Markdown cell below for interpretation.')

In [ ]:
# ROI comparison bar chart
channels = ['TV', 'Radio', 'Social_Media']
correlations = [
    df_clean[['TV',          'Sales']].corr().iloc[0, 1],
    df_clean[['Radio',       'Sales']].corr().iloc[0, 1],
    df_clean[['Social_Media','Sales']].corr().iloc[0, 1],
]

colors = ['#4C72B0' if c == max(correlations) else '#AEC7E8' for c in correlations]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(channels, correlations, color=colors, edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Pearson Correlation with Sales', fontsize=11)
ax.set_title('Channel Correlation with Sales\n(Proxy for Direct ROI Impact)', fontsize=13, fontweight='bold')
ax.axhline(0.8, color='gray', linestyle='--', linewidth=1, label='Strong correlation threshold (r=0.8)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('roi_comparison.png', dpi=150, bbox_inches='tight')
plt.show()